# Python Iterables, Iterators, Generators, and Coroutines

## 1. Iterable

An **iterable** is an object whose values can be accessed one at a time.

Examples:

```python
numbers = [1, 2, 3]
name = "Python"
items = {"a": 1, "b": 2}
```

These objects can be used in a `for` loop:

```python
for number in numbers:
    print(number)
```

An iterable usually provides an `__iter__()` method.

```python
iterator = iter(numbers)
```

Common iterables include:

- Lists
- Tuples
- Strings
- Dictionaries
- Sets
- Files
- Generator objects

## 2. Iterator

An **iterator** is an object that produces values one at a time.

An iterator follows two methods:

```python
__iter__()
__next__()
```

- `__iter__()` returns the iterator itself.
- `__next__()` returns the next value.
- When no values remain, `__next__()` raises `StopIteration`.

Example:

```python
numbers = [10, 20, 30]
iterator = iter(numbers)

print(next(iterator))  # 10
print(next(iterator))  # 20
print(next(iterator))  # 30
```

The next call raises `StopIteration`:

```python
next(iterator)
```

## 3. Iterable vs. Iterator

An iterable can create an iterator. An iterator keeps track of its current position.

```python
numbers = [1, 2, 3]       # Iterable
iterator = iter(numbers)  # Iterator
```

| Feature | Iterable | Iterator |
|---|---|---|
| Produces values | Yes | Yes |
| Has `__iter__()` | Usually | Yes |
| Has `__next__()` | Not necessarily | Yes |
| Can restart iteration | Usually | Usually no |
| Example | List, string | `iter(list)` |
| Stops with | Not directly | `StopIteration` |

A list can be iterated over multiple times:

```python
numbers = [1, 2, 3]

print(list(numbers))
print(list(numbers))
```

An iterator is normally consumed once:

```python
iterator = iter([1, 2, 3])

print(list(iterator))  # [1, 2, 3]
print(list(iterator))  # []
```

## 4. Creating a Custom Iterator

Create an iterator by defining `__iter__()` and `__next__()`.

```python
class CountUp:
    def __init__(self, maximum):
        self.current = 1
        self.maximum = maximum

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.maximum:
            raise StopIteration

        value = self.current
        self.current += 1
        return value
```

Usage:

```python
counter = CountUp(3)

for number in counter:
    print(number)
```

Output:

```text
1
2
3
```

The `for` loop internally performs roughly this process:

```python
iterator = iter(counter)

while True:
    try:
        value = next(iterator)
        print(value)
    except StopIteration:
        break
```

## 5. Generators

A **generator** is a simple way to create an iterator.

A generator function contains the `yield` keyword.

```python
def count_up(maximum):
    number = 1

    while number <= maximum:
        yield number
        number += 1
```

Usage:

```python
for number in count_up(3):
    print(number)
```

Output:

```text
1
2
3
```

Calling a generator function does not immediately run its body:

```python
generator = count_up(3)
```

The function runs when values are requested:

```python
print(next(generator))  # 1
print(next(generator))  # 2
print(next(generator))  # 3
```

## 6. `yield` vs. `return`

`return` ends a function completely.

```python
def get_value():
    return 10
```

`yield` temporarily pauses a generator and saves its state.

```python
def get_values():
    yield 10
    yield 20
    yield 30
```

When the generator is resumed, execution continues after the previous `yield`.

## 7. Generator State

Generators remember their local variables between executions.

```python
def numbers():
    print("Starting")
    yield 1
    print("Continuing")
    yield 2
    print("Finished")
```

Usage:

```python
generator = numbers()

print(next(generator))
# Starting
# 1

print(next(generator))
# Continuing
# 2

next(generator)
# Finished, then StopIteration
```

## 8. Generator Expressions

A **generator expression** is similar to a list comprehension but creates values lazily.

List comprehension:

```python
squares = [number ** 2 for number in range(5)]
```

Generator expression:

```python
squares = (number ** 2 for number in range(5))
```

The list stores all results immediately. The generator produces results when needed.

```python
for square in squares:
    print(square)
```

## 9. Memory Efficiency

Generators are useful for large data because they do not store every result in memory.

List:

```python
numbers = [number ** 2 for number in range(1_000_000)]
```

Generator:

```python
numbers = (number ** 2 for number in range(1_000_000))
```

The generator calculates one value at a time.

Generators are useful for:

- Large files
- Large calculations
- Data processing pipelines
- Infinite sequences
- Streaming data

## 10. Reading Large Files

Instead of loading the entire file:

```python
with open("large_file.txt") as file:
    lines = file.readlines()
```

Process one line at a time:

```python
with open("large_file.txt") as file:
    for line in file:
        print(line.strip())
```

File objects are iterable and produce lines lazily.

## 11. Infinite Generators

A generator can produce values indefinitely.

```python
def infinite_numbers():
    number = 0

    while True:
        yield number
        number += 1
```

Use a condition to stop consuming it:

```python
for number in infinite_numbers():
    if number == 5:
        break
    print(number)
```

## 12. `yield from`

`yield from` delegates iteration to another iterable or generator.

```python
def numbers():
    yield from [1, 2, 3]
    yield from [4, 5, 6]
```

Usage:

```python
print(list(numbers()))
```

Output:

```text
[1, 2, 3, 4, 5, 6]
```

Without `yield from`, yielding a list would produce the entire list as one value:

```python
def example():
    yield [1, 2, 3]
```

## 13. Generator Methods: `send()`

Generators can receive values using `.send()`.

```python
def receiver():
    print("Ready")
    value = yield
    print(f"Received: {value}")
```

Usage:

```python
generator = receiver()

next(generator)       # Starts the generator
generator.send("Hi")  # Sends a value into it
```

The first `next()` starts the generator because `.send()` cannot send a non-`None` value before the first `yield`.

## 14. Generator Return Values

A generator can use `return` to provide a final result.

```python
def calculate():
    yield 1
    yield 2
    return "Finished"
```

The return value is stored in the `StopIteration` exception:

```python
generator = calculate()

try:
    while True:
        print(next(generator))
except StopIteration as error:
    print(error.value)
```

## 15. Coroutines

A **coroutine** is a function whose execution can be paused and resumed, often to handle asynchronous tasks.

Modern Python coroutines are usually defined with `async def`.

```python
async def greet():
    print("Hello")
```

Calling the coroutine does not run it immediately:

```python
result = greet()
```

It must be executed using `await` inside another coroutine or with an event loop.

## 16. `async` and `await`

Use `await` to pause a coroutine while waiting for another asynchronous operation.

```python
import asyncio

async def greet():
    await asyncio.sleep(1)
    print("Hello")

asyncio.run(greet())
```

While `greet()` is waiting, the event loop can run other coroutines.

## 17. Asynchronous Example

```python
import asyncio

async def task(name, delay):
    print(f"{name} started")
    await asyncio.sleep(delay)
    print(f"{name} finished")

async def main():
    await asyncio.gather(
        task("Task A", 2),
        task("Task B", 1)
    )

asyncio.run(main())
```

Both tasks can make progress during waiting periods.

Expected order:

```text
Task A started
Task B started
Task B finished
Task A finished
```

## 18. Coroutine vs. Generator

Generators primarily produce values.

```python
def numbers():
    yield 1
    yield 2
```

Coroutines primarily perform or coordinate asynchronous work.

```python
async def download_data():
    await network_request()
```

| Feature | Generator | Coroutine |
|---|---|---|
| Defined with | `def` and `yield` | `async def` |
| Main purpose | Produce values lazily | Manage asynchronous operations |
| Paused with | `yield` | `await` |
| Started with | `next()` or a loop | `await` or an event loop |
| Common use | Data streams | Network, files, timers |
| Object type | Generator iterator | Coroutine object |

## 19. Asynchronous Generators

An **asynchronous generator** uses both `async def` and `yield`.

```python
import asyncio

async def async_numbers():
    for number in range(3):
        await asyncio.sleep(1)
        yield number
```

Consume it with `async for`:

```python
async def main():
    async for number in async_numbers():
        print(number)

asyncio.run(main())
```

An asynchronous generator can produce values over time without blocking the event loop.

## 20. `yield` and `await`

| Keyword | Meaning |
|---|---|
| `yield` | Produces a value and pauses a generator |
| `await` | Pauses a coroutine until an awaitable operation completes |
| `return` | Ends a function or coroutine |
| `async for` | Iterates over an asynchronous iterable |

## 21. Important Concepts

- An **iterable** can be looped over.
- An **iterator** produces values using `next()`.
- A **generator** is an easy way to create an iterator.
- `yield` pauses a generator and preserves its state.
- Generators are lazy and memory-efficient.
- A **coroutine** can pause and resume asynchronous work.
- `async def` creates a coroutine function.
- `await` pauses a coroutine without blocking other asynchronous tasks.
- `async for` consumes asynchronous iterables.
- `yield from` delegates values to another iterable.
- `.send()` sends data into a generator.

## 22. Quick Example

```python
import asyncio

def squares(numbers):
    for number in numbers:
        yield number ** 2

async def process():
    for square in squares(range(5)):
        print(square)
        await asyncio.sleep(0.1)

asyncio.run(process())
```

Here:

- `range(5)` is an iterable.
- `squares()` is a generator function.
- `yield` produces one square at a time.
- `process()` is a coroutine.
- `await` pauses the coroutine briefly.